<table>
  <tr>
    <td><div align="left"><font size="30">Getting started</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/master/figs/BDSimLogo_NoBackgnd%402x.png" width="300"></td>
  </tr>
</table>

(c) Peter Corke 2026

In [ ]:
# Install bdsim.  In JupyterLite, piplite checks the local wheel
# (built with 'make wheel') before falling back to PyPI.
# In a regular Jupyter server this is a no-op (piplite is not available).
try:
    import piplite
    await piplite.install('bdsim')
except ImportError:
    pass  # already installed (classic Jupyter)


In this notebook we will build a model of a simple dynamic system, a first-order system with a feedback loop, and simulate it.

# Getting started

We first need to import the `bdsim` (**b**lock **d**iagram **sim**ulation) package

In [ ]:
import bdsim

and then create an instance of the simulation environment that will run our model

In [ ]:
sim = bdsim.BDSim()

Next up we create an empty block diagram. Think of it as a blank "canvas" on which we will build our model 

In [ ]:
bd = sim.blockdiagram()

This is the standard boiler plate for every `bdsim` model you will create.  To summarize, the block diagram `bd` instance holds our model, and the `sim` instance will run our model.  This structure ensures a clear separation of the model and the runtime simulation environment.

We can print the block diagram object

In [ ]:
print(bd)

and can see, that at this stage, it has zero blocks and wires.



Now it's time to build our model and here's a sketch of what we'll build

![Block diagram sketch](../figs/bd1-sketch.png)

We can add the blocks in any order, but let's start with the plant itself, which is modeled by a first-order continuous-time transfer function

In [ ]:
plant = bd.LTI_SISO(0.5, [2, 1], name="plant")

We've invoked the `LTI_SISO` method of the `bd` instance.  This is really calling the constructor of a class called `LTI_SISO` and the result is an instance of that class and `plant` is a reference to that instance.  More about the details of this in section below.

The first argument is the coefficients of the numerator polynomial, a scalar in this case.  The second argument is the coefficients of the denominator polynomial.  The third argument is the name of the block which is how `bdsim` refers to it internally.  If no name is given a systematic default name will be assigned.

If it was a second-order system we could write something like

```
plant = bd.LTI_SISO(0.5, [2, -3, 4], name="plant")  # 2s^2 - 3s + 4
plant = bd.LTI_SISO(0.5, [2, (1,2), (1,3))], name="plant") # 2 (s+2)(s+3)
```

If we print the block we get a succinct summary of its key attributes

In [ ]:
print(plant)

It's name is "plant", it's class is "lti_siso" and is a subclass of "continuous" which represents continuous time dynamics.  The block has one input, one output, and one state variable.


Now let's define the summing junction and the gain block

In [ ]:
sum = bd.SUM("+-")
gain = bd.GAIN(10)

The summing junction has two inputs, the first is a "+" input, the second is a "-" input.  The gain block has a gain of 10.

Next we'll add a signal source to drive our system, we choose a unit step occuring at $t=1$

In [ ]:
demand = bd.STEP(T=1, name="demand")

and finally a 2 input "scope" block to display what's going on. For this example we'll plot them both against time in the same scope plot.

In [ ]:
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

The scope block will have two inputs because we specified two line styles (solid black and dashed red).  The legend will be placed in the lower right of the plot.

If we look at the block diagram object now

In [ ]:
print(bd)

we can see that it now has 4 blocks but there are no wires connecting them.

Let's connect them!  We do this by adding virtual wires. We connect the output of the summing junction to the input of the gain block by

In [ ]:

bd.connect(sum, gain)


which connects the single output port of `sum` to the single input port of `gain`.

The summing junction has two inputs and we use Python indexing notation to reference a specific port

In [ ]:
bd.connect(plant, sum[1])

which connects the single output port of `plant` to the second input port of `sum`.  It's the *second* because we use Python indexing which starts at zero.

In [ ]:
bd.connect(demand, sum[0], scope[1])

which connects the single output port of `demand` to two inputs: the first input of `sum` and the second input of `scope`.  This is *exactly* equivalent to writing

``` python
bd.connect(demand, sum[0])
bd.connect(demand, scope[1])
```

With reference to our sketch diagram, there are just two more wires to add

In [ ]:
bd.connect(gain, plant)
bd.connect(plant, scope[0])

It should be clear by now what these calls do.

You can create the blocks and wires in any order so long as the blocks that are joined by a wire already exist.

We can check back on our block diagram object

In [ ]:
print(bd)

and see that it now has 5 blocks and 6 wires.

Now we need to get our model ready to run.  The first step is to "compile" it

In [ ]:
bd.compile(verbose=True)

The compilation step performs lots of checks (the `verbose` options shows them all), and in this case it has succeeded.  Common errors are unconnected inputs, references to invalid blocks, or port indices that are out of bounds.

To see what `bdsim` has made of our model we can display some reports.  The first simply lists all the blocks and wires with their key characteristics

In [ ]:
bd.report()

Note that the wire list shows the type of signal on the wire.  This is determined by `compile` which simulates the model for a single time step and propagates the signals through the diagram.  The `demand` block has default "off" and "on" values that are `int`.

A more integrated view of the model comes from this single table report

In [ ]:
bd.report_summary()

which a block-input centric view.  The second last row, for example, is for the `SCOPE` block and shows that input 0 comes from "plant" and input 1 comes from "demand".

If you're having some errors at compile time these tables can help you cross-check what you've coded with your model.


Now it's time to run our model.  We'll see what happens over the first 5 seconds.

In [ ]:
out = sim.run(bd, T=5)

We see a classic first-order response to the unit step at time $t=1$ and considerable steady state error.

We could experiment with the model by:

* changing the demand signal (there is a `RAMP` block as well as `WAVEFORM` (sinusoid, square, triangle, sawtooth), `PIECEWISE` and `INTERP` blocks)
* changing the `GAIN`
* adding integral action, swapping the `GAIN` block for a `PID` block

The time series data from the simulation is available in the return value `out`

In [ ]:
print(out)

which is a "struct like" container object.  By default it contains just a time-series showing the evolution of the system state vector, one row per time step, one column per state.

`xnames` is a list of the state names, and its elements correspond to the columns of `x`.  In this case the state is $x_0$ of the "plant" box -- it's a first-order system with just one state variable.

We could plot the state against time

In [ ]:
import matplotlib.pyplot as plt
plt.plot(out.t, out.x);


To record what's happening on particular output ports we use a "watchlist". For example, to record the two signals that we plotted in the scope we can write

In [ ]:
out = sim.run(bd, T=5, watch=[demand, sum])


In [ ]:
print(out)

The watched ports, the output of the "plant" and "demand" blocks, appear as columns of the variable `y`. The label associated with each column is given by the corresponding names in the list `ynames`.


An alternative way to "watch" ports is to wire them to `WATCH` blocks, this will effectively add the ports they are connected to, to the watch list.

## A more Pythonic approach to model building

Let's try expressing this model in a different, perhaps more Pythonic, way.  We'll start over and create an empty block diagram

In [ ]:
bd = sim.blockdiagram()

and create just the core (non-arithmetic) blocks

In [ ]:
demand = bd.STEP(T=1, name="demand")
plant = bd.LTI_SISO(0.5, [2, 1], name="plant")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")  # , movie='eg1.mp4')

and now we'll wire them up using Python assignments and arithmetic operations

In [ ]:
scope[0] = plant
scope[1] = demand
plant[0] = 10 * (demand - plant)

Inputs are on the left-hand side, outputs on the right.  So we read the first line asthe input `scope[0]` is driven by `plant`. The third line creates a summing junction, a gain block, wires them all together and wires it to the first input of `plant`. 

In [ ]:
bd.compile(verbose=False)
bd.report_summary()

and we see *exactly* the same network topology as we had earlier!  Block names starting with underscore are *implictly* created by overloaded Python arithmetic operators.

This requires just 6 lines of code compared to 10 for the explicit wiring approach. Whether you find it more elegant depends on your own preferences, one is not better than the other. You can mix both approaches - a wire created by assignment is just the same as one created by `bd.connect(...)`.

# Advanced topics

## Ports and vectors

In the example above the signals were scalars, ints and floats.  So long as the objects support the `*`, `+` and `-` operators then they can be fed into the `GAIN` and `SUM` blocks.  This means that we can pass in NumPy arrays of arbitrary shape, so long as they conform.  The `GAIN` block can multiple an array input by a scalar, and the output would be a vector of the same width.  However the gain itself can be an array and we rely on NumPy semantics to determine the shape of the output array.  Note that the `GAIN` block has an option to select between pre- and post-multiplication of the input by the gain value.

A `SCOPE` block can accept the signals to plot on multiple scalar input ports, or it can plot the elements of a vector on a single input port.

The `DEMUX` block expands an N-element 1d array on the input port to N scalar output ports.  The `MUX` block does the opposite.

## Scripts and command line options

Notebooks are convenient, but more commonly you would create a Python script that contains your model definition using `bdsim`.  See for example [`examples/eg1.py`](https://github.com/petercorke/bdsim/blob/master/examples/eg1.py).

You can exert considerable control over the model using command line switches


`bdsim` has many run-time options.  Options, in decreasing precedence, are given by the command line, the arguments to `BDSim` and environment variables. The allowable command line options are:

In [ ]:
sim.options.print_help()

Within your program you can see the current options state by

In [ ]:
print(sim)

For example, you can disable graphics

``` shell
$ ./eg1.py -g
```

turn off all chatter

``` shell
$ ./eg1.py -q
```

save the output to a JSON file
``` shell
$ ./eg1.py -j out.json
```

or a Python pickle file
``` shell
$ ./eg1.py -p out.pkl
```

or change parameter values
``` shell
$ ./eg1.py -g
```

# Animations and movies

# Dynamic block loading

All blocks are classes.  For example, you can see the source code of the `GAIN` block [here](https://github.com/petercorke/bdsim/blob/9603af34d51d3ec4ed98b424013a242a60eda165/bdsim/blocks/functions.py#L250).  The class is actually called `Gain`.  There is a hierarchy of block classes: `Gain` → `FunctionBlock` → `Block`.

When we call

``` python
sim = bdsim.BDSim()
```

as well as handling all the options, it loads all the Python files that define blocks:



In [ ]:
sim.blocks()

These include the "core" blocks from `bdsim` but also blocks defined in companion packages as the Robotics Toolbox for Python, Machine Vision Toolbox for Python and Spatial Math.

When we create a new blockdiagram

``` python
bd = sim.blockdiagram()
```

it turns the `bd` instance into a block "factory".  The `__init__` method of class `Gain` is turned into the method call `bd.GAIN`.  Blocks have a number of other methods like `output`, `deriv` and `next` which are used by the runtime simulation engine to simulate the model.



The ones prefixed by `bdsim.blocks` are shipped with the bdsim package but the Robotics Toolbox and MachineVision Toolbox for Python also define blocks, and if you have them installed they might be found and listed.

Note that the block names shown here are in camel case whereas later we will be referring to with uppercase versions of these names.


